In [41]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


In [42]:
df=pd.read_csv('daily-minimum-temperatures-in-me.csv',encoding="utf-8", sep=',', on_bad_lines='skip')
df.head()

,Date,"Daily minimum temperatures in Melbourne, Australia, 1981-1990"
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8


In [43]:
df.columns

Index(['Date', 'Daily minimum temperatures in Melbourne, Australia, 1981-1990'], dtype='object')

In [44]:
df.rename(columns={"Daily minimum temperatures in Melbourne, Australia, 1981-1990":"Temp"},inplace=True)

In [45]:
df.head()

,Date,Temp
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8


In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3650 entries, 0 to 3649
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Date    3650 non-null   object
 1   Temp    3650 non-null   object
dtypes: object(2)
memory usage: 57.2+ KB


In [47]:
df.isnull().sum()

Date    0
Temp    0
dtype: int64

In [48]:
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

In [49]:
df['Temp']=pd.to_numeric(df['Temp'],errors='coerce')
df.dropna(inplace=True)

In [50]:
values=df["Temp"].values

In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3647 entries, 1981-01-01 to 1990-12-31
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Temp    3647 non-null   float64
dtypes: float64(1)
memory usage: 57.0 KB


In [52]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [53]:
scaler = MinMaxScaler()
df["Temp"]=scaler.fit_transform(df[["Temp"]])


In [123]:
import numpy as np

def create_sequences(data, time_steps=30):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:i+time_steps])
        y.append(data[i+time_steps])
    return np.array(X), np.array(y)

X, y = create_sequences(df["Temp"], 30)

In [124]:
X_train, X_test,y_train,y_test= train_test_split(X,y, test_size=0.2,shuffle=False)

In [135]:
import tensorflow
from keras.models import Sequential
from keras.layers import SimpleRNN,Dense,Dropout
from keras.callbacks import EarlyStopping

In [136]:
model=Sequential()

In [137]:
model.add(SimpleRNN(1024,return_sequences=True,activation="tanh",input_shape=(50,1)))
model.add(Dropout(0.5))
model.add(SimpleRNN(512,return_sequences=True,activation="tanh"))
model.add(Dropout(0.5))
model.add(SimpleRNN(256,return_sequences=True,activation="tanh"))
model.add(Dropout(0.5))
model.add(SimpleRNN(128,return_sequences=True,activation="tanh"))
model.add(Dropout(0.5))
model.add(SimpleRNN(64,return_sequences=True,activation="tanh"))
model.add(Dropout(0.5))
model.add(SimpleRNN(32,return_sequences=False,activation="tanh"))

model.add(Dropout(0.5))
model.add(Dense(1))

In [138]:
model.compile(optimizer="adam",loss="mse")

In [139]:
model.fit(X_train,y_train,epochs=50,validation_data=(X_test,y_test),callbacks=EarlyStopping(patience=3))

Epoch 1/50


91/91 ━━━━━━━━━━━━━━━━━━━━ 97s 844ms/step - loss: 1.6131 - val_loss: 0.1768
Epoch 2/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 81s 894ms/step - loss: 0.9302 - val_loss: 0.7453
Epoch 3/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 116s 1s/step - loss: 0.5406 - val_loss: 0.0392
Epoch 4/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 122s 1s/step - loss: 0.3632 - val_loss: 0.2666
Epoch 5/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 112s 722ms/step - loss: 0.2343 - val_loss: 0.0564
Epoch 6/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 86s 757ms/step - loss: 0.1469 - val_loss: 0.0446


In [140]:
y_pred=model.predict(X_test)

23/23 ━━━━━━━━━━━━━━━━━━━━ 14s 436ms/step


In [141]:
y_pred=scaler.inverse_transform(y_pred)
y_test_actual=scaler.inverse_transform(y_test.reshape(-1,1))

In [142]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test_actual, y_pred)
print("Mean Squared Error:", mse)

Mean Squared Error: 30.87798616455918
